# PFAS × ACS — Data Loading & Join

Reads the **two PFAS halves** from `data/raw/` (detections + non-detections, produced by
`pfas_to_parquet.py --kind detect|nondetect`), **verifies they don't overlap**, concatenates
them, then joins to ACS ZCTA demographics (**ZIP → ZCTA**) and writes the merged
**ZCTA × month** dataset to `data/processed/`.

Non-detects were substituted to `0` upstream, so `mean_result_ngL` is the mean over **all**
samples — a sampled-but-clean ZCTA-month is `0`, not `NaN`. `is_detection` is preserved so
detection *rates* stay recoverable. `city` and a **rurality** proxy (population density) are
attached from the SimpleMaps `uszips.csv` crosswalk.

In [15]:
import os, glob
import numpy as np, pandas as pd
import pyarrow.parquet as pq

RAW = os.environ.get("RAW_DIR", "../data/raw")
OUT = os.environ.get("OUT_DIR", "../data/processed")
os.makedirs(OUT, exist_ok=True)

# The two PFAS halves are named like *detect*.parquet (detects_detect / nondetects_nondetect);
# the ACS file like acs*.parquet. Matching by name keeps stray parquets (old exports) out.
pfas_paths = sorted(glob.glob(os.path.join(RAW, "*detect*.parquet")))
acs_cands  = sorted(glob.glob(os.path.join(RAW, "acs*.parquet")))
assert pfas_paths, f"No PFAS halves (*detect*.parquet) found in {RAW}"
assert acs_cands,  f"No ACS parquet (acs*.parquet) found in {RAW}"
acs_path = acs_cands[0]

# sanity-check column signatures so a mis-named file can't slip through
for p in pfas_paths:
    n = set(pq.read_schema(p).names)
    assert {"ZIP Codes Served", "is_detection"} <= n, f"{os.path.basename(p)} lacks PFAS columns"
assert "ZCTA" in set(pq.read_schema(acs_path).names), f"{os.path.basename(acs_path)} lacks 'ZCTA'"

ignored = [p for p in sorted(glob.glob(os.path.join(RAW, "*.parquet")))
           if p not in pfas_paths + [acs_path]]
print("PFAS halves:", ", ".join(os.path.basename(p) for p in pfas_paths))
print("ACS        :", os.path.basename(acs_path))
if ignored:
    print("IGNORED (not loaded):", ", ".join(os.path.basename(p) for p in ignored))

PFAS halves: detects_detect.parquet, nondetects_nondetect.parquet
ACS        : acs5_2023_zcta.parquet
IGNORED (not loaded): 30bb373d-f158-4d30-a06c-f729eaea6a99.parquet


## 1. Load & concatenate the two PFAS halves

In [16]:
pfas = pd.concat([pd.read_parquet(p) for p in pfas_paths], ignore_index=True)
print(f"{len(pfas):,} rows | detections: {int(pfas['is_detection'].fillna(False).sum()):,}"
      f" | systems: {pfas['PWS ID'].nunique():,} | contaminants: {pfas['Contaminant'].nunique()}")
pfas.head(3)

1,889,931 rows | detections: 37,546 | systems: 10,297 | contaminants: 30


,PWS ID,PWS Name,Size,Facility ID,Facility Name,Facility Water Type,Sample Point ID,Sample Point Name,Sample Point Type,Collection Date,...,Longitude,Population Served,Population Served Year,Most Recent Sample,Potential PFAS Sources,PFAS Treatment,UCMR Cycle,Results,UCMR_RAA_Active,is_detection
0,010106001,Mashantucket Pequot Water System,Large,00006,MPTN WTP,GU-Groundwater Under Influence of Surface Water,TP1,Entry point to Dist. System,EP,2023-08-09,...,-71.972828,37807.0,2023.0,None,None,None,UCMR 5 (2023-2025),1,1.0,False
1,010106001,Mashantucket Pequot Water System,Large,00006,MPTN WTP,GU-Groundwater Under Influence of Surface Water,TP1,Entry point to Dist. System,EP,2023-08-09,...,-71.972828,37807.0,2023.0,None,None,None,UCMR 5 (2023-2025),1,1.0,False
2,010106001,Mashantucket Pequot Water System,Large,00006,MPTN WTP,GU-Groundwater Under Influence of Surface Water,TP1,Entry point to Dist. System,EP,2023-08-09,...,-71.972828,37807.0,2023.0,None,None,None,UCMR 5 (2023-2025),1,1.0,False


## 1b. Verify the halves don't overlap (no double-counting)

A UCMR result row is uniquely identified by PWS + sample + contaminant (+ MRL + ZIPs). If the
two downloads were split cleanly by result-vs-MRL there should be **zero** duplicate keys; any
duplicates mean a system-level download pulled the same rows into both halves, so we drop them.

In [17]:
key_cols = [c for c in ["PWS ID", "Sample ID", "Contaminant", "Collection Date",
                        "Minimum Reporting Level (ng/L)", "ZIP Codes Served"]
            if c in pfas.columns]
comp = pfas["is_detection"].fillna(False).value_counts().rename({True: "detect", False: "non-detect"})
print("detection-flag composition:", comp.to_dict())
print("key columns:", key_cols)

dup = pfas.duplicated(subset=key_cols, keep=False)
print(f"duplicate-key rows across halves: {int(dup.sum()):,}")
if dup.any():
    print("  -> halves OVERLAP (likely a system-level download). Dropping duplicates (keep first).")
    pfas = pfas.drop_duplicates(subset=key_cols).reset_index(drop=True)
    print(f"  -> {len(pfas):,} rows after de-dup")
else:
    print("  -> clean: the two halves are disjoint.")

detection-flag composition: {'non-detect': 1852385, 'detect': 37546}
key columns: ['PWS ID', 'Sample ID', 'Contaminant', 'Collection Date', 'Minimum Reporting Level (ng/L)', 'ZIP Codes Served']
duplicate-key rows across halves: 21,239
  -> halves OVERLAP (likely a system-level download). Dropping duplicates (keep first).
  -> 1,873,595 rows after de-dup


## 2. Explode `ZIP Codes Served` → one row per served ZIP (ZCTA candidate)

In [18]:
# keep only the columns the aggregation needs, before exploding (memory!)
pfas["is_detection"] = pfas["is_detection"].astype("boolean").fillna(False).astype(bool)

KEEP = ["ZIP Codes Served", "PWS ID", "PWS Name", "State Territory or Tribe", "Contaminant",
        "Analytical Result Value (ng/L)", "Minimum Reporting Level (ng/L)",
        "is_detection", "Collection Date"]

missing_zip = int(pfas["ZIP Codes Served"].isna().sum())
small = pfas.loc[pfas["ZIP Codes Served"].notna(), KEEP].copy()
small["zcta"] = small["ZIP Codes Served"].str.split(";")
small = small.drop(columns="ZIP Codes Served").explode("zcta")
small["zcta"] = small["zcta"].str.strip().str.zfill(5)
small = small[small["zcta"].str.fullmatch(r"\d{5}")]
small["month"] = small["Collection Date"].dt.to_period("M").dt.to_timestamp()

print(f"no-ZIP rows excluded: {missing_zip:,}")
print(f"exploded rows: {len(small):,} | unique ZCTAs served: {small['zcta'].nunique():,}")
print(f"exploded frame: {small.memory_usage(deep=True).sum()/1e6:,.0f} MB")

no-ZIP rows excluded: 112,489
exploded rows: 9,483,141 | unique ZCTAs served: 19,963
exploded frame: 3,138 MB


## 3. Aggregate to ZCTA × month, then join ACS + city + rurality

In [19]:
acs = pd.read_parquet(acs_path)
acs["ZCTA"] = acs["ZCTA"].astype("string").str.zfill(5)

# aggregate to ZCTA x month FIRST, then attach ACS (so the big frame is never merged).
# non-detects are 0 upstream -> this mean is over ALL samples (clean ZCTA-month -> 0, not NaN).
pfas_agg = small.groupby(["zcta", "month"]).agg(
    mean_result_ngL=("Analytical Result Value (ng/L)", "mean"),   # mean over all samples (non-detects = 0)
    n_records=("Analytical Result Value (ng/L)", "size"),
    n_detections=("is_detection", "sum"),
    n_systems=("PWS ID", "nunique"),
    n_contaminants=("Contaminant", "nunique"),
    pws_names=("PWS Name", lambda s: "; ".join(sorted(s.dropna().unique()))),
    state=("State Territory or Tribe",
           lambda s: s.dropna().mode().iat[0] if not s.dropna().mode().empty else None),
).reset_index()
pfas_agg["any_detection"] = pfas_agg["n_detections"] > 0

zcta_month = pfas_agg.merge(acs, how="left", left_on="zcta", right_on="ZCTA")

# city + rurality via ZIP->ZCTA crosswalk (SimpleMaps uszips.csv); ZCTA code == representative ZIP
xwalk = pd.read_csv(os.path.join(OUT, "uszips.csv"), dtype=str,
                    usecols=["zip", "city", "density", "population"])
xwalk["zip"] = xwalk["zip"].str.zfill(5)
for c in ["density", "population"]:
    xwalk[c] = pd.to_numeric(xwalk[c], errors="coerce")
xwalk = xwalk.dropna(subset=["zip"]).drop_duplicates("zip")
zcta_month = zcta_month.merge(xwalk, how="left", left_on="zcta", right_on="zip").drop(columns="zip")
zcta_month["city"] = zcta_month["city"].fillna("Unknown city")
zcta_month = zcta_month.rename(columns={"density": "pop_density", "population": "zip_population"})

# rurality proxy from population density (people / sq mi); thresholds heuristic, adjust to taste
def _rurality(d):
    if pd.isna(d): return "Unknown"
    if d < 500:    return "Rural"
    if d < 3000:   return "Suburban"
    return "Urban"
zcta_month["rurality"] = zcta_month["pop_density"].apply(_rurality)

matched = int(zcta_month["ZCTA"].notna().sum())
print(f"ACS ZCTAs: {len(acs):,}")
print(f"{len(zcta_month):,} ZCTA-months | matched to ACS: {matched:,} ({matched/len(zcta_month):.1%})"
      f" | city matched: {(zcta_month['city'] != 'Unknown city').mean():.1%}"
      f" | rurality known: {(zcta_month['rurality'] != 'Unknown').mean():.1%}")
print("rurality mix:", zcta_month["rurality"].value_counts(dropna=False).to_dict())
print(f"unique ZCTAs: {zcta_month['zcta'].nunique():,} | "
      f"unmatched ZCTAs: {zcta_month.loc[zcta_month['ZCTA'].isna(),'zcta'].nunique():,}")
zcta_month.head()

ACS ZCTAs: 33,772
117,379 ZCTA-months | matched to ACS: 108,632 (92.5%) | city matched: 92.6% | rurality known: 92.5%
rurality mix: {'Rural': 66977, 'Suburban': 35240, 'Unknown': 8756, 'Urban': 6406}
unique ZCTAs: 19,963 | unmatched ZCTAs: 1,782


,zcta,month,mean_result_ngL,n_records,n_detections,n_systems,n_contaminants,pws_names,state,any_detection,...,pct_black_nh,pct_asian_nh,pct_amerind_nh,pct_hispanic,pct_nonwhite,pct_renter,city,zip_population,pop_density,rurality
0,00601,2023-03-01,0.0,29,0,1,29,GARZAS,PR,False,...,0.131571,0.11363,0.0,99.455774,99.700975,30.458029,Adjuntas,16669.0,99.9,Rural
1,00601,2023-05-01,0.0,29,0,1,29,GARZAS,PR,False,...,0.131571,0.11363,0.0,99.455774,99.700975,30.458029,Adjuntas,16669.0,99.9,Rural
2,00601,2023-08-01,0.0,29,0,1,29,GARZAS,PR,False,...,0.131571,0.11363,0.0,99.455774,99.700975,30.458029,Adjuntas,16669.0,99.9,Rural
3,00601,2023-11-01,0.0,29,0,1,29,GARZAS,PR,False,...,0.131571,0.11363,0.0,99.455774,99.700975,30.458029,Adjuntas,16669.0,99.9,Rural
4,00601,2025-02-01,0.0,29,0,1,29,ADJUNTAS URBANO,PR,False,...,0.131571,0.11363,0.0,99.455774,99.700975,30.458029,Adjuntas,16669.0,99.9,Rural


In [20]:
zcta_month.isna().sum() / len(zcta_month)

zcta                    0.000000
month                   0.000000
mean_result_ngL         0.000000
n_records               0.000000
n_detections            0.000000
n_systems               0.000000
n_contaminants          0.000000
pws_names               0.000000
state                   0.000000
any_detection           0.000000
ZCTA                    0.074519
GEO_ID                  0.074519
NAME                    0.074519
total_pop               0.074519
median_hh_income        0.108972
poverty_universe        0.074519
poverty_below           0.074519
race_universe           0.074519
white_nh                0.074519
black_nh                0.074519
amerindian_alaska_nh    0.074519
asian_nh                0.074519
hispanic                0.074519
median_age              0.084990
tenure_universe         0.074519
renter_occupied         0.074519
pct_below_poverty       0.087912
pct_white_nh            0.082519
pct_black_nh            0.082519
pct_asian_nh            0.082519
pct_amerin

## 4. Save the full merged dataset

In [21]:
out_path = os.path.join(OUT, "pfas_acs_zcta_month.parquet")
zcta_month.to_parquet(out_path, compression="zstd", index=False)
print(f"wrote {out_path}  ({os.path.getsize(out_path)/1e6:.2f} MB, "
      f"{len(zcta_month):,} ZCTA-months x {zcta_month.shape[1]} cols)")

wrote ../data/processed\pfas_acs_zcta_month.parquet  (4.83 MB, 117,379 ZCTA-months x 38 cols)


## To use the final dataset

```
import pandas as pd
df = pd.read_parquet("data/processed/pfas_acs_zcta_month.parquet")
```

New columns this version adds: `pop_density`, `zip_population`, `rurality`
(Rural / Suburban / Urban / Unknown). `mean_result_ngL` now includes non-detects as 0,
so a value of 0 = sampled-but-clean; ZCTAs absent from the file = never sampled.